In [1]:
import pandas as pd

train_file_path = './open/train/train.csv'
market_file_path = './open/train/meta/TRAIN_전국도매_2018-2021.csv'
sanjae_file_path = './open/train/meta/TRAIN_산지공판장_2018-2021.csv'

train_df = pd.read_csv(train_file_path)
whole_df = pd.read_csv(market_file_path)
local_df = pd.read_csv(sanjae_file_path)

In [3]:
whole_df.head(5)

,시점,시장코드,시장명,품목코드,품목명,품종코드,품종명,총반입량(kg),총거래금액(원),평균가(원/kg),...,저가(20%) 평균가,중간가(원/kg),최저가(원/kg),최고가(원/kg),경매 건수,전순 평균가격(원) PreVious SOON,전달 평균가격(원) PreVious MMonth,전년 평균가격(원) PreVious YeaR,평년 평균가격(원) Common Year SOON,연도
0,201801상순,100000,*전국도매시장,501,감자,50124,깐감자,20.0,86520,4326.000000,...,4326.000000,4326.000000,4326.0,4326.000000,1,0.000000,4009.000000,0.000000,0.000000,2018
1,201801상순,100000,*전국도매시장,501,감자,50121,돼지감자,12380.0,11650810,941.099354,...,545.105717,1010.000000,200.0,3000.000000,117,11213.358450,9174.196723,8167.895632,0.000000,2018
2,201801상순,100000,*전국도매시장,501,감자,50110,자주감자,240.0,158400,660.000000,...,500.000000,550.000000,500.0,1000.000000,7,12553.279352,12612.216445,24990.324897,18483.961304,2018
3,201801상순,100000,*전국도매시장,501,감자,50111,가을감자,10.0,37500,3750.000000,...,3700.000000,3750.000000,3700.0,3800.000000,2,24929.463415,40365.081269,0.000000,0.000000,2018
4,201801상순,100000,*전국도매시장,501,감자,50199,기타감자,1367301.3,2403199462,1757.622451,...,955.289668,1360.453431,0.0,10581.081081,872,30806.779529,27661.150770,23741.953223,19340.121989,2018


In [3]:
local_df.head(0)

,시점,공판장코드,공판장명,품목코드,품목명,품종코드,품종명,등급코드,등급명,총반입량(kg),...,평균가(원/kg),중간가(원/kg),최저가(원/kg),최고가(원/kg),경매 건수,전순 평균가격(원) PreVious SOON,전달 평균가격(원) PreVious MMonth,전년 평균가격(원) PreVious YeaR,평년 평균가격(원) Common Year SOON,연도


In [2]:
whole_df['year'] = whole_df['시점'].str.slice(0,4).astype(int)
whole_df['month'] = whole_df['시점'].str.slice(4,6).astype(int)
whole_df['순'] = whole_df['시점'].str.slice(6)

def soon_to_day(순):
    mapping ={'상순': 10, '중순': 20, '하순': 30}
    return mapping.get(순,15)

whole_df['day'] = whole_df['순'].apply(soon_to_day)
whole_df['date'] = pd.to_datetime(whole_df[['year', 'month', 'day']], errors='coerce')
whole_df = whole_df.sort_values('date').reset_index(drop=True)

In [3]:
# sorting and grouping
whole_df['period'] = whole_df['year'].astype(str) + whole_df['month'].astype(str).str.zfill(2) + whole_df['순']
whole_df.sort_values(by=['품목명', '품종명', 'date', '순'], inplace=True)


In [4]:
# Assign an order number to each period within each group
whole_df['order'] = whole_df.groupby(['품목명', '품종명']).cumcount()


## calculating lagged avg price

In [5]:
# previous 순 avg price
whole_df['전순 평균가(원/kg) Calculated'] = whole_df.groupby(['품목명', '품종명'])['평균가(원/kg)'].shift(1)

In [7]:
# previous month avg price
def get_prev_date(row):
    prev_month = row['date'] - pd.DateOffset(months=1)
    last_day_prev_month = prev_month.days_in_month
    return prev_month.replace(day=min(row['day'], last_day_prev_month))

whole_df['전달_date'] = whole_df.apply(get_prev_date, axis = 1)
whole_df = whole_df.merge(
    whole_df[['품목명', '품종명', 'date', '평균가(원/kg)']],
    left_on=['품목명', '품종명', '전달_date'],
    right_on=['품목명', '품종명', 'date'],
    how='left',
    suffixes=('', '_전달')
)

whole_df.rename(columns={'평균가(원/kg)_전달': '전달 평균가(원/kg) Calculated'}, inplace=True)

In [8]:
def get_previous_year_date(row):
    prev_year = row['date'] - pd.DateOffset(years=1)
    return prev_year.replace(day=row['day'])

whole_df['전년_date'] = whole_df.apply(get_previous_year_date, axis=1)

# Merge to get the previous year's average price
whole_df = whole_df.merge(
    whole_df[['품목명', '품종명', 'date', '평균가(원/kg)']],
    left_on=['품목명', '품종명', '전년_date'],
    right_on=['품목명', '품종명', 'date'],
    how='left',
    suffixes=('', '_전년')
)

# Rename the merged column
whole_df.rename(columns={'평균가(원/kg)_전년': '전년 평균가(원/kg) Calculated'}, inplace=True)


MemoryError: Unable to allocate 41.6 GiB for an array with shape (5579135631,) and data type int64

In [ ]:
import numpy as np

# Compare previous '순'
whole_df['전순 평균가격(원) Matches'] = np.isclose(
    whole_df['전순 평균가(원/kg) Calculated'],
    whole_df['전순 평균가격(원) PreVious SOON'],
    atol=1e-2
)

# Compare previous month
whole_df['전달 평균가격(원) Matches'] = np.isclose(
    whole_df['전달 평균가(원/kg) Calculated'],
    whole_df['전달 평균가격(원) PreVious MMonth'],
    atol=1e-2
)

# Compare previous year
whole_df['전년 평균가격(원) Matches'] = np.isclose(
    whole_df['전년 평균가(원/kg) Calculated'],
    whole_df['전년 평균가격(원) PreVious YeaR'],
    atol=1e-2
)


## checking infulence

In [ ]:
price_columns = ['저가(20%) 평균가', '중간가(원/kg)', '최저가(원/kg)', '최고가(원/kg)']
previous_avg_price_columns = ['전순 평균가격(원) PreVious SOON', '전달 평균가격(원) PreVious MMonth', '전년 평균가격(원) PreVious YeaR']

# Select relevant columns
correlation_df = whole_df[price_columns + previous_avg_price_columns + ['평균가(원/kg)']]

# Compute correlation matrix
correlation_matrix = correlation_df.corr()

print(correlation_matrix)
